In [7]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, BertConfig, BertModel

In [8]:
#settings
seed=42
train_fraction=0.8
validation_fraction=0.2
batch_size=128
Bert_Model  =  "google/bert_uncased_L-2_H-256_A-4"
tokenizer = AutoTokenizer.from_pretrained(Bert_Model, local_files_only=True)
tokenizer.model_max_length = 10**9
device = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = device == "cuda"
# we define negative and positive labels as 0 and 1 respectively, DONT CHANGE THIS
train_df=pd.read_csv("train.csv")
submission_data=pd.read_csv ("test.csv")
train_data = train_df[["review", "label"]]
train_data["label_id"] = train_data["label"].map({"negative": 0, "positive": 1}).astype(int) 
total_data=len(train_data)
rng = np.random.default_rng(seed)
indices = np.arange(total_data)
rng.shuffle(indices)
N_train = int(total_data * train_fraction)
N_val = int(total_data * validation_fraction)
# indices
train_indices = indices[:N_train]
val_indices = indices[N_train:N_train + N_val]

# I dont know how to make this part parallel, maybe you can change this. but the speed is enough, I think...
texts = train_data["review"].astype(str).tolist()
labels = train_data["label_id"].tolist()
train_texts = [texts[i] for i in train_indices]
val_texts = [texts[i] for i in val_indices]

train_labels = [labels[i] for i in train_indices]
val_labels = [labels[i] for i in val_indices]




In [9]:

class ReviewCLSDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = labels
        self.max_length = 512
        self.encodings = tokenizer(list(texts), truncation=True, max_length=512, padding="max_length")

    def _encode_for_cls(self, text):
        token_ids = tokenizer.encode(str(text), add_special_tokens=True)
        # Due that our bert only has 512 tokens, 
        # we need to truncate long reviews. 
        # for short reviews, we can just use the standard [CLS] review [SEP] format.
        if len(token_ids) <= self.max_length - 2:
            input_ids =token_ids

        else:
        # for long reviews, we keep the head and tail parts, and truncate the middle part.
        # this is my (Zikai Zhao) personally choice, you can change this if you want.
        # but the whole review is long, if you decide to just think about the head part
        # the accuracy may not be high.
            head_len = self.max_length // 2
            tail_len = self.max_length - head_len
            head_ids = token_ids[:head_len]
            tail_ids = token_ids[-tail_len:]
            input_ids = (head_ids+tail_ids)
        
        token_type_ids = [0] * len(input_ids)
        attention_mask = [1] * len(input_ids)
        pad_len = self.max_length - len(input_ids)

        return {
            "input_ids": input_ids + [tokenizer.pad_token_id] * pad_len,
            "attention_mask": attention_mask + [0] * pad_len,
            "token_type_ids": token_type_ids + [0] * pad_len,
        }

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {
                "input_ids": torch.tensor(self.encodings["input_ids"][idx], dtype=torch.long),
                "attention_mask": torch.tensor(self.encodings["attention_mask"][idx], dtype=torch.long),
                "token_type_ids": torch.tensor(self.encodings["token_type_ids"][idx], dtype=torch.long),
                "labels": torch.tensor(self.labels[idx], dtype=torch.long), }
        return item
    
train_dataset = ReviewCLSDataset(train_texts, train_labels)
val_dataset = ReviewCLSDataset(val_texts, val_labels)


In [ ]:
#our model
from pytest import param


class BertSmallCLSClassifier(nn.Module):
    def __init__(self, model_name, dropout=0.2, num_labels=2,hidden_size=64):
        super().__init__()

        self.config = BertConfig.from_pretrained(
            model_name,
            local_files_only=True,
            num_labels=num_labels,
            id2label={0: "negative", 1: "positive"},
            label2id={"negative": 0, "positive": 1},
        )

        self.bert = BertModel.from_pretrained(
            model_name,
            config=self.config,
            local_files_only=True,
        )
        for param in self.bert.parameters():
            param.requires_grad = True 
        self.classifier = nn.Sequential(
            nn.Linear(self.config.hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_labels),
        )

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(input_ids=input_ids,attention_mask=attention_mask,token_type_ids=token_type_ids)
        cls_output = outputs.last_hidden_state[:, 0, :] # we take the [CLS] token's output
        result = self.classifier(cls_output)
        return result

model = BertSmallCLSClassifier(Bert_Model).to(device)
total_params = sum(p.numel() for p in model.parameters())
print("total params:", f"{total_params:,}")


total params: 5,187,394


/Users/dongchuanmiao/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [11]:
# adjust batch size according to your GPU memory
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

val_loader = DataLoader(val_dataset,batch_size=128,shuffle=False)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    [
        {"params": model.bert.parameters(), "lr": 2e-5},
        {"params": model.classifier.parameters(), "lr": 1e-4},
    ],
    weight_decay=0,
)



In [46]:

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

for epoch in range(10):
    model.train()
    total_train_loss = 0
    train_correct = 0
    train_total = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        pred = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        loss = criterion(pred, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        predictions = torch.argmax(pred, dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)
    avg_train_loss = total_train_loss / len(train_loader)
    train_acc = train_correct / train_total
    train_losses.append(avg_train_loss)
    train_accuracies.append(train_acc)
    # validation
    model.eval()
    total_val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels = batch["labels"].to(device)

            pred = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )

            loss = criterion(pred, labels)

            total_val_loss+=loss.item()

            predictions=torch.argmax(pred, dim=1)
            val_correct+=(predictions == labels).sum().item()
            val_total+=labels.size(0)
    avg_val_loss= total_val_loss/len(val_loader)
    val_acc =val_correct/val_total
    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)
    print("epoch:", epoch + 1)
    print("train loss:", avg_train_loss)
    print("train accuracy:", train_acc)
    print("val loss:", avg_val_loss)
    print("val accuracy:", val_acc)
    print()